In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [6]:
# data: https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset
data = pd.read_csv('../datasets/insurance.csv')
data.columns = [col.strip() for col in data.columns.values]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [7]:
data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [20]:
data_encoded = pd.get_dummies(data=data, columns=['sex', 'smoker', 'region'], drop_first=True)
data_encoded.corr()['charges'].sort_values(ascending=False)

charges             1.000000
smoker_yes          0.787251
age                 0.299008
bmi                 0.198341
region_southeast    0.073982
children            0.067998
sex_male            0.057292
region_northwest   -0.039905
region_southwest   -0.043210
Name: charges, dtype: float64

### Model

In [71]:
# feature selection
numeric_features = ['age', 'bmi', 'children']
category_features = ['sex', 'smoker', 'region']

X = data.drop('charges', axis=1)
y = data['charges']

# data split
X_train_, X_test_, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# encoding categorical feature
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded_train = encoder.fit_transform(X_train_[category_features])
encoded_test = encoder.transform(X_test_[category_features])

encoded_features = encoder.get_feature_names_out(category_features)
encoded_X_train = pd.DataFrame(encoded_train, columns=encoded_features, index=X_train_.index).reset_index(drop=True)
encoded_X_test = pd.DataFrame(encoded_test, columns=encoded_features, index=X_test_.index).reset_index(drop=True)


# # easier approach for encoding
# X_train = pd.get_dummies(X_train, columns=category_features, drop_first=True)
# X_test = pd.get_dummies(X_test, columns=category_features, drop_first=True)
# X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# scaling numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_.reset_index(drop=True)[numeric_features])
X_test_scaled = scaler.transform(X_test_.reset_index(drop=True)[numeric_features])

scaled_X_train = pd.DataFrame(X_train_scaled, columns=numeric_features)
scaled_X_test = pd.DataFrame(X_test_scaled, columns=numeric_features)


# combine categorical and numerical features together
X_train = pd.concat([
    scaled_X_train,
    encoded_X_train
], axis=1)

X_test = pd.concat([
    scaled_X_test,
    encoded_X_test
], axis=1)

# model 
model = LinearRegression()
model.fit(X_train, y_train)

# predict
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# feature_importance
feature_importance = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_
})
feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

print("="*50)
print("FEATURE IMPORTANCE")
print("="*50)
print(feature_importance)



FEATURE IMPORTANCE
            feature   coefficient  abs_coefficient
4        smoker_yes  23651.128856     23651.128856
0               age   3614.975415      3614.975415
1               bmi   2036.228123      2036.228123
7  region_southwest   -809.799354       809.799354
6  region_southeast   -657.864297       657.864297
2          children    516.890247       516.890247
5  region_northwest   -370.677326       370.677326
3          sex_male    -18.591692        18.591692


In [72]:
# performance metrics
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

# print(f"\n📊 Cross Validation:")
# cv_scores = cross_val_score(model, X, y, cv=5)
# print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 78.4% of the variation in house price of unit area
   • Only 21.6% of variation is unexplained (due to other factors)
Training R²: 0.7417
Test R²: 0.7836
✅ Good: Training and testing R² are similar - no overfitting


📊 MSE:
Training MSE: 37277681.70
Test MSE: 33596915.85


📊 RMSE
   • Predictions are off by ±5796.28 on average
   • In other words, 68% of predictions fall within 5796.28 of actual value
   • 95% of predictions fall within 11592.57 of actual value
Training RMSE: 6105.55
Testing RMSE: 5796.28


📊 MAE:
Training MAE: 4208.23
Testing MAE: 4181.19


In [68]:
# ========================
# using pipeline (to work with cv r2 or metrics like this we have to create a pipline)
# ========================

# feature selection
numeric_features = ['age', 'bmi', 'children']
category_features = ['sex', 'smoker', 'region']

X = data.drop('charges', axis=1)
y = data['charges']

# data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaling and encoding features
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), category_features)
])

# model
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
model.fit(X_train, y_train)

# predict
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# feature_importance
preprocessor = model.named_steps['preprocessor']
encoder = preprocessor.named_transformers_['cat']
categorical_features = encoder.get_feature_names_out(['sex', 'smoker', 'region'])

features = list(numeric_features) + list(categorical_features)
coefficients = model.named_steps['regressor'].coef_

feature_importance = pd.DataFrame({
    'feature': features,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

print("="*50)
print("FEATURE IMPORTANCE")
print("="*50)
print(feature_importance)

FEATURE IMPORTANCE
            feature   coefficient  abs_coefficient
4        smoker_yes  23651.128856     23651.128856
0               age   3614.975415      3614.975415
1               bmi   2036.228123      2036.228123
7  region_southwest   -809.799354       809.799354
6  region_southeast   -657.864297       657.864297
2          children    516.890247       516.890247
5  region_northwest   -370.677326       370.677326
3          sex_male    -18.591692        18.591692


In [69]:
# performance metrics
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

print(f"\n📊 Cross Validation:")
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 78.4% of the variation in house price of unit area
   • Only 21.6% of variation is unexplained (due to other factors)
Training R²: 0.7417
Test R²: 0.7836
✅ Good: Training and testing R² are similar - no overfitting


📊 Cross Validation:
Cross-validation R²: 0.747 (+/- 0.025)


📊 MSE:
Training MSE: 37277681.70
Test MSE: 33596915.85


📊 RMSE
   • Predictions are off by ±5796.28 on average
   • In other words, 68% of predictions fall within 5796.28 of actual value
   • 95% of predictions fall within 11592.57 of actual value
Training RMSE: 6105.55
Testing RMSE: 5796.28


📊 MAE:
Training MAE: 4208.23
Testing MAE: 4181.19
